# E01 — Energy-side equivalence with the edge data center study

| | |
|---|---|
| **Question** | Given identical load, PV and battery inputs, do Vessim and the edge study produce the same grid exchange and battery state? |
| **Hypotheses** | H4 (Vessim's energy-side abstractions are sound and at least as clean as ours); partly H3 (control expressiveness) |
| **Branch** | `exp/e01-energy-equivalence` |
| **Reference** | Edge study scenario `s1_2_battery_pv_large` (one spring day, 15-min physical step), plus a variant with three times the PV area |

**Approach.** The edge study is run once per case in its own environment (`external/.venv-edge`);
its recorded site demand, PV and battery results are resampled to 15 minutes and cached in `data/`.
Vessim is then driven with exactly these traces and four configurations:

| Vessim configuration | Battery model | Battery control |
|---|---|---|
| Vessim default | `SimpleBattery` (built-in, lossless) | `DefaultDispatchPolicy` (built-in, greedy) |
| Vessim default + losses | `EfficiencyBattery` (custom, edge study parameters) | `DefaultDispatchPolicy` |
| Vessim + edge rule + losses | `EfficiencyBattery` | edge study online rule as a custom `Controller` |
| Vessim, optimal schedule replay | `EfficiencyBattery` | edge study offline-optimal battery schedule as a setpoint signal |

## 1. Setup

In [1]:
import time
import pandas as pd
from IPython.display import display

from vessim_eval import RESULTS_DIR
from vessim_eval import plotting as vp
from vessim_eval.edge_reference import load_reference, scenario_config
from vessim_eval.scenarios import (BatteryParams, SiteInputs, agreement, energy_summary,
                                   run_site)
import vessim

print("Vessim version:", vessim.__version__ if hasattr(vessim, "__version__") else
      __import__("importlib.metadata").metadata.version("vessim"))

Vessim version: 0.16.0


In [2]:
# ---- Parameters -------------------------------------------------------------
SCENARIO = "s1_2_battery_pv_large"
REFERENCE_CSV = "e01_edge_s1_2_15min.csv"         # edge study, S1.2
REFERENCE_PV_X3_CSV = "e01_edge_s1_2_pv_x3_15min.csv"  # edge study, S1.2 with 3x PV area
STEP_S = 900
OUT = RESULTS_DIR / "E01"

site_cfg = scenario_config(SCENARIO)["data_centers"][0]
battery = BatteryParams.from_edge_config(site_cfg["battery_config"])
base_load_kw = site_cfg["hardware"]["base_load"]

ref = load_reference(REFERENCE_CSV)
ref_x3 = load_reference(REFERENCE_PV_X3_CSV)
inputs = SiteInputs.from_reference(ref, base_load_kw)
inputs_x3 = SiteInputs.from_reference(ref_x3, base_load_kw)
battery

BatteryParams(capacity_kwh=500, soc_init=0.5, soc_min=0.15, soc_max=0.9, max_charge_kw=250.0, max_discharge_kw=250.0, eff_ch=0.95, eff_dis=0.95)

### Input traces (edge study S1.2)
Exogenous inputs passed to Vessim as `Trace` signals: IT hardware, cooling, building and PV active power.

In [3]:
fig = vp.stacked_panels(
    [("Active power (kW)", {"IT hardware": ref.it_kw, "Cooling": ref.cooling_kw,
                             "Photovoltaic generation": ref.pv_kw}),
     ("Active power (kW)", {"Building": ref.building_kw})],
    title="Site inputs, edge study S1.2 (15-min samples)")
for tr, key, dash in zip(fig.data, ["it", "cooling", "pv", "building"],
                         ["solid", "dash", "dot", "dashdot"]):
    tr.line.color, tr.line.dash = vp.COMPONENT_COLORS[key], dash
fig.show()

## 2. Run

In [4]:
CONFIGS = {
    "Vessim default": dict(battery_model="simple", control="greedy"),
    "Vessim default + losses": dict(battery_model="losses", control="greedy"),
    "Vessim + edge rule + losses": dict(battery_model="losses", control="edge_rule"),
    "Vessim, optimal schedule replay": dict(battery_model="losses", control="schedule",
                                            schedule_discharge_kw=ref.lp_battery_setpoint_kw),
}
runs, walls = {}, {}
for name, kw in CONFIGS.items():
    runs[name], walls[name] = run_site(inputs, battery, **kw)

# Case with 3x PV area: the edge online rule actually charges the battery.
runs_x3 = {}
runs_x3["Vessim + edge rule + losses"], walls["Vessim + edge rule + losses, PV x3"] = run_site(
    inputs_x3, battery, battery_model="losses", control="edge_rule")
runs_x3["Vessim default + losses"], _ = run_site(inputs_x3, battery, battery_model="losses",
                                                  control="greedy")

# Input sampling: same configuration, time-weighted means instead of step-start samples.
runs_mean, _ = run_site(SiteInputs.from_reference(ref, base_load_kw, mean=True), battery,
                        battery_model="losses", control="edge_rule")

pd.Series(walls, name="Wall time (s)").to_frame().style.format("{:.4f}")

,Wall time (s)
Vessim default,0.0083
Vessim default + losses,0.0063
Vessim + edge rule + losses,0.0052
"Vessim, optimal schedule replay",0.0055
"Vessim + edge rule + losses, PV x3",0.0055


Edge study wall time for the same simulated day: **≈ 165–200 s**. 96 % of it is the greedy job
scheduler (2 555 calls); the physical co-simulation (mosaik, hardware, cooling chain, recorder)
takes ≈ 6 s. Not like-for-like — Vessim only integrates the energy balance here. See the E00 report
and E09.

## 3. Results

### 3.1 Energy balance check
Vessim enforces an exact power balance at every step; the residual should be at floating-point level.

In [5]:
pd.Series({n: r["balance_residual_kw"].abs().max() for n, r in runs.items()},
          name="Maximum power balance residual (kW)").to_frame().style.format("{:.2e}")

,Maximum power balance residual (kW)
Vessim default,9.09e-13
Vessim default + losses,9.09e-13
Vessim + edge rule + losses,9.09e-13
"Vessim, optimal schedule replay",9.09e-13


### 3.2 Energy indicators

**Conventions.** All series below use: grid power positive = import, battery power positive =
discharge, state of charge at the *start* of each 15-min step. The edge study records measured SOC
at the *end* of a step (but its optimal schedule at the start), and logs only battery discharge;
charging power is recovered from its demand balance. The helper below converts accordingly.

In [6]:
def edge_frame(r, lp=False):
    # Edge results in the standardized column layout used for Vessim runs.
    if lp:
        return pd.DataFrame({"grid_kw": r.lp_grid_kw, "battery_kw": r.lp_battery_setpoint_kw,
                             "soc": r.lp_soc})
    return pd.DataFrame({"grid_kw": r.grid_import_kw, "battery_kw": r.battery_net_out_kw,
                         "soc": r.soc.shift(1).fillna(battery.soc_init)})

edge = edge_frame(ref)
edge_lp = edge_frame(ref, lp=True)
edge_x3 = edge_frame(ref_x3)

kpi = pd.DataFrame({"Edge study": energy_summary(edge, STEP_S),
                    **{n: energy_summary(r, STEP_S) for n, r in runs.items()},
                    "Edge study, optimal schedule": energy_summary(edge_lp, STEP_S)})
vp.format_table(kpi.T, 1)

,Grid import energy (kWh),Grid export energy (kWh),Peak grid import (kW),Battery charge energy (kWh),Battery discharge energy (kWh),Minimum state of charge (%),Maximum state of charge (%)
Edge study,"110,157.5",0.0,"6,255.3",0.0,0.0,50.0,50.0
Vessim default,"109,982.5",0.0,"6,255.3",0.0,175.0,15.0,50.0
Vessim default + losses,"109,991.3",0.0,"6,255.3",0.0,166.2,15.0,50.0
Vessim + edge rule + losses,"110,157.5",0.0,"6,255.3",0.0,0.0,50.0,50.0
"Vessim, optimal schedule replay","110,198.1",0.0,"6,005.3",416.0,375.4,15.0,59.0
"Edge study, optimal schedule","110,198.1",0.0,"6,005.3",416.0,375.4,15.0,59.0


In [7]:
fig = vp.grouped_bars(
    kpi.loc[["Battery discharge energy (kWh)", "Battery charge energy (kWh)"]],
    title="Battery energy throughput over the day, S1.2", y_title="Energy (kWh)")
fig.show()

### 3.3 Online control: edge study vs Vessim (S1.2)

In [8]:
fig = vp.stacked_panels(
    [("Grid active power (kW, + = import)",
      {"Edge study": edge.grid_kw, **{n: runs[n].grid_kw for n in list(CONFIGS)[:3]}}),
     ("Battery state of charge (%)",
      {"Edge study": edge.soc * 100, **{n: runs[n].soc * 100 for n in list(CONFIGS)[:3]}})],
    title="Grid exchange and battery state of charge, S1.2: edge study vs Vessim")
fig.show()
vp.save(fig, OUT / "fig_online_control_s1_2")

### 3.4 Online rule in an active regime (PV area x3)

In [9]:
fig = vp.stacked_panels(
    [("Photovoltaic generation (kW)", {"Edge study": ref_x3.pv_kw}),
     ("Battery active power (kW, + = discharge)",
      {"Edge study": edge_x3.battery_kw, **{n: r.battery_kw for n, r in runs_x3.items()}}),
     ("Battery state of charge (%)",
      {"Edge study": edge_x3.soc * 100, **{n: r.soc * 100 for n, r in runs_x3.items()}})],
    title="Edge battery rule re-implemented in Vessim, PV area x3")
fig.show()
vp.save(fig, OUT / "fig_edge_rule_pv_x3")

In [10]:
kpi_x3 = pd.DataFrame({"Edge study": energy_summary(edge_x3, STEP_S),
                       **{n: energy_summary(r, STEP_S) for n, r in runs_x3.items()},
                       "Edge study, optimal schedule": energy_summary(edge_frame(ref_x3, lp=True), STEP_S)})
vp.format_table(kpi_x3.T, 1)

,Grid import energy (kWh),Grid export energy (kWh),Peak grid import (kW),Battery charge energy (kWh),Battery discharge energy (kWh),Minimum state of charge (%),Maximum state of charge (%)
Edge study,"103,904.1",0.0,"6,255.3",210.5,0.0,50.0,90.0
Vessim + edge rule + losses,"103,904.1",0.0,"6,255.3",210.5,0.0,50.0,90.0
Vessim default + losses,"103,527.4",0.0,"6,255.3",0.0,166.2,15.0,50.0
"Edge study, optimal schedule","103,734.2",0.0,"6,005.3",416.0,375.4,15.0,59.0


### 3.5 Offline-optimal schedule replay

In [11]:
fig = vp.stacked_panels(
    [("Battery active power (kW, + = discharge)",
      {"Edge study, optimal schedule": edge_lp.battery_kw,
       "Vessim, optimal schedule replay": runs["Vessim, optimal schedule replay"].battery_kw}),
     ("Battery state of charge (%)",
      {"Edge study, optimal schedule": edge_lp.soc * 100,
       "Vessim, optimal schedule replay": runs["Vessim, optimal schedule replay"].soc * 100}),
     ("Grid active power (kW, + = import)",
      {"Edge study, optimal schedule": edge_lp.grid_kw,
       "Vessim, optimal schedule replay": runs["Vessim, optimal schedule replay"].grid_kw})],
    title="Edge study offline-optimal battery schedule replayed in Vessim's energy balance")
fig.show()
vp.save(fig, OUT / "fig_lp_replay")

### 3.6 Agreement summary

In [12]:
pairs = {
    "Grid power, edge rule (S1.2)": (edge.grid_kw, runs["Vessim + edge rule + losses"].grid_kw),
    "Grid power, edge rule (PV x3)": (edge_x3.grid_kw, runs_x3["Vessim + edge rule + losses"].grid_kw),
    "Battery power, edge rule (PV x3)": (edge_x3.battery_kw, runs_x3["Vessim + edge rule + losses"].battery_kw),
    "State of charge, edge rule (PV x3)": (edge_x3.soc, runs_x3["Vessim + edge rule + losses"].soc),
    "Grid power, optimal replay": (edge_lp.grid_kw, runs["Vessim, optimal schedule replay"].grid_kw),
    "State of charge, optimal replay": (edge_lp.soc, runs["Vessim, optimal schedule replay"].soc),
    "Grid power, Vessim default": (edge.grid_kw, runs["Vessim default"].grid_kw),
}
agree = pd.DataFrame({k: agreement(a, b, STEP_S) for k, (a, b) in pairs.items()}).T
agree.style.format("{:.3g}")

,Maximum absolute difference,Mean absolute difference,Daily energy difference (kWh)
"Grid power, edge rule (S1.2)",1e-06,9.38e-08,-1.75e-06
"Grid power, edge rule (PV x3)",1e-06,9.54e-08,-1.71e-06
"Battery power, edge rule (PV x3)",1.58e-07,1.64e-09,-3.95e-08
"State of charge, edge rule (PV x3)",4.65e-07,1.65e-08,-1.64e-07
"Grid power, optimal replay",1e-06,1.15e-07,-1.25e-06
"State of charge, optimal replay",4.49e-07,4.17e-08,9.23e-07
"Grid power, Vessim default",250,7.29,-175


In [13]:
fig = vp.parity_plot(
    edge.grid_kw,
    {"Vessim default": runs["Vessim default"].grid_kw,
     "Vessim + edge rule + losses": runs["Vessim + edge rule + losses"].grid_kw},
    title="Grid active power per 15-min step, S1.2", axis_label="Grid active power (kW)")
fig.show()

In [14]:
diffs = agree["Maximum absolute difference"].clip(lower=1e-12).to_frame("Maximum absolute difference")
fig = vp.grouped_bars(diffs, title="Largest per-step deviation from the edge study (log scale)",
                      y_title="Maximum absolute difference (kW or state-of-charge fraction)",
                      log_y=True, horizontal=True, height=460)
fig.update_traces(texttemplate="%{x:.1e}")
fig.show()
vp.save(fig, OUT / "fig_agreement")

### 3.7 Input sampling: step-start samples vs time-weighted means
The edge study's energy supply and grid accounting take the demand value at the start of each
15-min step and hold it, while IT hardware power changes every 30 s. Vessim does the same when fed
step samples. Feeding time-weighted step means instead shows the size of this effect.

In [15]:
fig = vp.line_comparison(
    {"Sampled at step start": ref.it_kw, "Time-weighted step mean": ref.it_mean_kw},
    title="IT hardware active power: step-start sample vs time-weighted mean (edge study S1.2)",
    y_title="Active power (kW)")
fig.show()
e_sampled = energy_summary(runs["Vessim + edge rule + losses"], STEP_S)["Grid import energy (kWh)"]
e_mean = energy_summary(runs_mean, STEP_S)["Grid import energy (kWh)"]
print(f"Grid import, sampled inputs: {e_sampled:,.0f} kWh; time-weighted inputs: {e_mean:,.0f} kWh; "
      f"difference {e_mean - e_sampled:,.0f} kWh ({(e_mean / e_sampled - 1) * 100:.2f} %)")

Grid import, sampled inputs: 110,158 kWh; time-weighted inputs: 110,534 kWh; difference 376 kWh (0.34 %)


## 4. Evaluation

- **Equivalence holds.** With identical inputs and the edge study's battery model and rule
  re-implemented as Vessim extensions, grid power, battery power and state of charge agree with
  the edge study to < 1e-6 (CSV rounding) in both the idle (S1.2) and the active (PV x3) regime,
  and for the replayed offline-optimal schedule. The energy accounting of both tools is the same.
- **Vessim's power balance is exact by construction** (residual ~1e-12 kW at every step); an
  infeasible balance raises an error instead of silently drifting.
- **Out of the box Vessim does not reproduce our study.** `SimpleBattery` has no charge/discharge
  efficiency, no upper SOC limit and a single symmetric C-rate; `DefaultDispatchPolicy` is greedy
  (discharges 175 kWh in the first hour and sits at minimum SOC for the rest of the day — no peak
  shaving). Matching required ~120 lines of custom code: a `Storage` subclass, a `DispatchPolicy`
  and a `Controller`.
- **Control interface constraints.** A `DispatchPolicy` receives only the net power imbalance,
  the interval length and grid signals — no timestamp and no actor states. A schedule had to be
  passed in as a *grid signal*; the edge rule (which needs PV and building load separately) had to
  be a `Controller`, whose decisions take effect one step later. That one-step delay happens to
  match the edge study's time-shifted mosaik connection.
- **Edge study observations surfaced by the comparison:**
  (i) S1.2's online battery rule never operates (PV peak 594 kW < base load ≈ 1 170 kW);
  (ii) only battery discharge is recorded — charging is hidden in total demand;
  (iii) measured SOC is end-of-step while the optimal schedule's SOC is start-of-step;
  (iv) grid accounting samples demand at the step start, under-counting daily grid import by
  ≈ 0.34 % relative to time-weighted inputs.
- **Speed.** Vessim integrates the day in ≈ 5–10 ms vs ≈ 165–200 s for the full edge
  co-simulation. 96 % of the edge time is its greedy job scheduler; without it the edge
  co-simulation takes ≈ 6 s.